<h2 style="color:#4A148C; margin-bottom:6px;">
  05 – BERT Base (Uncased) for Multi-Label Emotion Classification
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  This notebook implements a pretrained transformer-based model using
  <strong>BERT (bert-base-uncased)</strong> for multi-label emotion
  classification on short English text.
</p>

<p style="font-size:16px;">
  <strong>Objective:</strong>
  To fine-tune a pretrained language model to predict five emotions
  (<em>anger, fear, joy, sadness, surprise</em>) simultaneously, and to
  evaluate its performance using the Macro F1-score.
</p>

<p style="font-size:16px;">
  The model is trained using a supervised fine-tuning approach with a
  sigmoid-based output layer for multi-label prediction. Validation
  performance is monitored to apply early stopping and select the best
  model checkpoint, which is later used for inference in a separate
  notebook.
</p>


In [1]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

W&B login successful!


In [2]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 05a – BERT Base (uncased) Training for Multi-Label Emotion Classification which does the below 
# - Logs to Weights & Biases
# - Saves best model
# - Uploads model to KaggleHub

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm
import wandb
import kagglehub
import os

LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

# -------------------------
# 0. W&B Init
# -------------------------
wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="05a_bert_base_uncased_training",
    config={
        "model_name": "bert-base-uncased",
        "max_len": 80,
        "batch_size": 16,
        "lr": 2e-5,
        "epochs": 25,
        "val_split": 0.1,
        "threshold": 0.5,
        "patience": 3,
        "random_state": 42
    }
)

config = wandb.config

# For reproducibility 
torch.manual_seed(config.random_state)
np.random.seed(config.random_state)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# 1. Load Data
# -------------------------
train_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
test_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")

tokenizer = BertTokenizer.from_pretrained(config.model_name)
MAX_LEN = config.max_len

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, mode="train"):
        self.df = df
        self.tokenizer = tokenizer
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]["text"])
        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.mode == "train":
            labels = torch.tensor(
                self.df.iloc[idx][LABELS].values.astype("float32")
            )
            item["labels"] = labels
        return item

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=config.val_split,
    random_state=config.random_state
)

train_dataset = EmotionDataset(train_df_split, tokenizer, mode="train")
val_dataset   = EmotionDataset(val_df_split, tokenizer, mode="train")

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=config.batch_size)

# -------------------------
# 2. Model Definition
# -------------------------
class BertForMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.linear = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.linear(pooled)
        return logits  # raw logits

model = BertForMultiLabel(config.model_name, len(LABELS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.lr)

# -------------------------
# 3. Training Loop + Early Stopping
# -------------------------
best_val_f1 = -1.0
patience = config.patience
wait = 0
best_model_path = "/kaggle/working/bert_base_uncased_best.pt"

for epoch in range(config.epochs):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} - Training")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_labels, val_preds = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.epochs} - Validation"):
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attn)
            probs = torch.sigmoid(logits).cpu().numpy()

            val_preds.append(probs)
            val_labels.append(labels.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    pred_bin = (val_preds > config.threshold).astype(int)

    val_f1_macro = f1_score(val_labels, pred_bin, average="macro")
    val_accuracy = accuracy_score(val_labels, pred_bin)

    print(
        f"Epoch {epoch+1}/{config.epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Macro F1: {val_f1_macro:.4f} "
        f"- Val Accuracy: {val_accuracy:.4f}"
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_f1_macro": val_f1_macro,
        "val_accuracy": val_accuracy
    })

    # Early stopping logic (based on Macro F1)
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        wait = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

print("Best validation Macro F1:", best_val_f1)

# -------------------------
# 4. Upload Best Model to KaggleHub
# -------------------------
KAGGLE_USERNAME = "sharmilamamani" 
MODEL = "emotion-bert"
FRAMEWORK = "pytorch"
VARIATION = "bert-base-uncased-v2"

handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

print("Uploading model to KaggleHub...")
repo = kagglehub.model_upload(
    handle,
    best_model_path,
    version_notes="BERT base uncased fine-tuned for multi-label emotion classification"
)

print("Uploaded model to KaggleHub.")
print("Handle:", handle)
print("Repo info:", repo)

wandb.finish()


2025-11-29 18:03:57.311126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764439437.512803      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764439437.568647      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Epoch 9/25 - Train Loss: 0.0164 - Val Macro F1: 0.8622 - Val Accuracy: 0.7277
Early stopping.
Best validation Macro F1: 0.864556156722523
Uploading model to KaggleHub...
Uploading Model https://www.kaggle.com/models/sharmilamamani/emotion-bert/pytorch/bert-base-uncased-v2 ...
Starting upload for file /kaggle/working/bert_base_uncased_best.pt


Uploading: 100%|██████████| 438M/438M [00:03<00:00, 128MB/s] 

Upload successful: /kaggle/working/bert_base_uncased_best.pt (418MB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/sharmilamamani/emotion-bert/pytorch/bert-base-uncased-v2
Uploaded model to KaggleHub.
Handle: sharmilamamani/emotion-bert/pytorch/bert-base-uncased-v2
Repo info: None


epoch,▁▂▃▄▅▅▆▇█
train_loss,█▅▃▂▂▁▁▁▁
val_accuracy,▁▄▆▇▇█▇██
val_f1_macro,▁▅▇▇█████
epoch,9
train_loss,0.0164
val_accuracy,0.72767
val_f1_macro,0.86222


<hr/>

<h3 style="color:#D35400;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Model:</strong> BERT Base (uncased) fine-tuned for multi-label emotion classification
</p>

<p style="font-size:16px;">
  <strong>Final Training Epoch:</strong> 5 - bert-base-uncased-v1<br/>
  Train Loss: <strong>0.05502</strong>
</p>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.85319</strong><br/>
  Accuracy: <strong>0.70132</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.846</strong>
</p>

<p style="font-size:16px;">
  <strong>Final Training Epoch:</strong> 9 - bert-base-uncased-v2<br/>
  Train Loss: <strong>0.0164</strong>
</p>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.862</strong><br/>
  Accuracy: <strong>0.728</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.854</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> GPU (NVIDIA T4 &times; 2)</li>
  <li><strong>Estimated Runtime:</strong> ~8–10 minutes (including training and validation)</li>
  <li><strong>Frameworks:</strong> PyTorch, HuggingFace Transformers</li>
</ul>

<p style="font-size:16px;">
  <strong>Observations:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li>
    Fine-tuned BERT achieves the highest performance among all evaluated
    models, substantially improving Macro F1 score compared to
    traditional and scratch-based neural approaches.
  </li>
  <li>
    The small difference between validation Macro F1 and Kaggle public
    leaderboard score indicates strong generalization to unseen test
    data.
  </li>
  <li>
    Pretrained contextual representations enable BERT to effectively
    capture subtle and overlapping emotional cues in short text samples.
  </li>
</ul>
